# Biohub Cell Tracking - SUBMISSION notebook (offline, no-internet safe)

Model-agnostic submission: runs a trained **3D U-Net detector** (heatmap regression) with the
**v2 LB-best detection front-end** + **rule-based two-pass motion-aware linking + gap-closing +
short-track filtering**, writes `/kaggle/working/submission.csv`.

**Current LB best = submission "Version 8" = LB 0.844** (converged v1 detector + v2 pipeline, divisions OFF).
That +0.017 jump over v2's 0.827 came from **nothing but better-trained detector weights** (60-epoch cosine
continuation), and it **matches the reference UNet's 0.843** using our *same* v2 linking -> **detector QUALITY
is a real, open lever** (density and divisions are both closed). See `docs/experiments.md` (v1b) for the trail.

## Pick the detector: `DETECTOR = 'v5_base24_nores'` for this A/B (config cell)

The active detector is **`v5_base24_nores`**: full-res anisotropic UNet, `BASE=24`, InstanceNorm, no residual.
It runs through the **identical LB-best v2 post-processing** as Version 8, so this submission is a clean
single-variable detector swap vs the converged v1 baseline (**LB 0.844**). Attach the v5 weights dataset; the
preferred filename is `v5_base24_nores_best.pt`, and the notebook also accepts the current training output name
`v5_UNet_res_best.pt`.

Other retained detector modes: `v1` (converged full-res base16, Version 8) and `v4` (isotropic pooled grid).
For all modes, keep `HM_THR=0.3` for the first A/B; if node counts change materially, recalibrate density as a
separate submission.

**Everything else is the LB-best v2 combination:** `HM_THR=0.3`, `min_distance=3` (v1/v5) / `1` (v4 pooled),
**NMS OFF**, two-pass Hungarian (tight 6 / **loose 8** um) + `close_gaps` + `prune_isolated` +
`filter_short_tracks(4)`, **`DIV_ENABLE=False`** (post-hoc divisions proven net LB-negative, Version 7 = 0.822).

**Why divisions are OFF.** The post-hoc `detect_divisions` step (v3) is edge-safe locally but proven **net
LB-negative**: submission "Version 7" (v2 base + divisions ON) scored **LB 0.822 vs 0.827**. The
`0.1*division_jaccard` bonus is negligible (~+0.0002), while the added mother->second-daughter edges -- which
look free against the sparse *local* GT -- become real edge-FPs on the *densely*-labeled hidden test set.

**Why v2 detection density (not v2.5b high-recall).** v2.5b (HM_THR 0.15 + min_dist 1 + NMS 3.0) LB-regressed
to **0.786** vs v2's **0.827** under identical linking (local tie 0.8588) -> the high-recall detection DENSITY
is the LB cost (adjusted over-prediction penalty + FP edges on the dense hidden GT that local eval can't see).

LB progression: **v1 NN 0.768 -> v2 two-pass linking 0.827 -> v2.5 relaxed 0.776 -> v2.5b high-recall 0.786
-> v2+v3 divisions 0.822 -> converged v1 + v2 pipeline 0.844 (BEST)**.

**Before submitting:**
1. Add Input: the competition data, the `zarr3-offline` wheel dataset, and your **weights dataset**. Set
   `DETECTOR` to match those weights. Attach ONLY the intended `.pt` (or set `WEIGHTS_PATH`) so auto-find
   doesn't pick the wrong one -- v4 needs symmetric STRIDES + the pool front-end, and v5 needs `BASE=24`;
   a detector/weights mismatch will error on `load_state_dict` (conv shapes differ).
2. Turn **Internet OFF** and Run All to VERIFY the offline zarr install + weight load end-to-end.
3. Sanity: each movie should print non-zero nodes/edges and `0 divisions` (DIV_ENABLE=False). A movie
   printing 0 nodes/edges signals a detection/weights problem.
4. Save Version, then Submit. For v5, compare to the 0.844 baseline to read the detector swap as a single
   attributable change.

**Density-penalty note:** HM_THR 0.3 + NMS off is the LB-calibrated density on the full-res grid. Do NOT lower
`HM_THR`, re-enable NMS, or turn divisions back on without an LB test -- all three are proven LB-negative.

In [ ]:
# Offline zarr>=3 install: auto-find the attached wheel dataset (no internet needed).
import glob, os, subprocess, sys
def find_wheeldir():
    for whl in glob.glob('/kaggle/input/**/zarr*.whl', recursive=True):
        return os.path.dirname(whl)
    return None
try:
    import zarr
    assert int(zarr.__version__.split('.')[0]) >= 3
    print('zarr already present:', zarr.__version__)
except Exception:
    wd = find_wheeldir()
    assert wd, 'No zarr wheel found under /kaggle/input -- attach the zarr3-offline dataset!'
    print('installing zarr offline from', wd)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', wd, 'zarr'])
    import zarr
    print('installed zarr:', zarr.__version__)

In [ ]:
import numpy as np, pandas as pd, time
from dataclasses import dataclass
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max

# ---------------- CONFIG (BEST COMBINATION: converged v1 detector + v2 detection + v2 linking, divisions OFF) ----------------
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um per voxel (anisotropic)
SCALE = VOXEL                                  # alias used by the rule-based linkers

# ================= DETECTOR VARIANT (the ONLY detector-side variable for this A/B) =================
# 'v5_base24_nores' = full-res anisotropic UNet, BASE=24, InstanceNorm, no residual.
#        weights file: v5_base24_nores_best.pt preferred; v5_UNet_res_best.pt accepted for current run output.
# 'v1' = converged full-res anisotropic UNet, BASE=16 -> submission Version 8 = LB 0.844.
#        weights file: v1_UNet_best.pt  (dataset biohub-v1-unet-base16-heatmap)
# 'v4' = isotropic 64^3 pooled-grid UNet (XY max-pool x4, symmetric strides).
#        weights file: v4_UNet_iso_best.pt  (dataset biohub-v4-unet-base16-iso-heatmap)
DETECTOR = 'v5_base24_nores'   # active detector A/B vs Version 8's v1 LB 0.844

# detection -- v2 preset (FROZEN = LB-BEST). Shared by both detectors; HM_THR is the density knob.
# v2.5b high-recall detection (HM_THR 0.15 + min_dist 1 + NMS 3.0) LB-regressed to 0.786 vs v2's 0.827 under
# IDENTICAL linking (local tie 0.8588) -> the detection density is the LB cost. Keep HM_THR 0.3, NMS off.
NORM_PCT     = (50.0, 99.8)   # percentile normalisation, must match training
HM_THR       = 0.3            # heatmap peak threshold (v2 value; v2.5's 0.15 over-detected -> LB penalty)
MIN_DISTANCE = 3              # peak_local_max separation (full-res v1/v5 voxels; overridden to 1 for v4 pooled grid)
MAX_PEAKS    = 200000         # per-frame cap
NMS_UM       = 0.0            # physical (um) NMS radius; 0 = OFF (v2 had no NMS; physical_nms is a no-op)
REFINE       = True           # intensity-weighted centre-of-mass sub-voxel refine
REFINE_WIN   = (1, 3, 3)      # v1 full-res CoM window (overridden to (1,5,5) for v4)

# linking + post-processing -- v2 (LB-proven precision; v2.5's relaxed values dropped LB 0.827 -> 0.776)
TIGHT_UM       = 6.0          # link_twopass tight gate
LOOSE_UM       = 8.0          # link_twopass loose gate (v2 value)
VEL_BLEND      = 0.5          # constant-velocity prediction blend
CLOSE_GAPS     = True         # bridge single-frame detection gaps
MAX_GAP        = 1
GAP_DIST_UM    = 6.0
FILTER_MIN_LEN = 4            # drop components shorter than this (v2 value)
PRUNE_ISOLATED = True

# post-hoc DIVISION detection (v3) -- DISABLED: proven LB-NEGATIVE, kept only for reference.
# The two-pass Hungarian is 1:1 so it never emits a division; detect_divisions adds mother->second-daughter
# edges after linking. It is EDGE-safe locally (added edges land on unmatched orphans) but net LB-negative:
#   * on v2.5b high-recall detection -> division-J 0.0024 (~400:1 FP), ~0 bonus;
#   * on the v2 base -> submission "Version 7" scored LB 0.822 vs v2's 0.827 (-0.005). The 0.1*div-J bonus
#     is negligible while the added M->D2 edges become real edge-FPs on the DENSE hidden GT (there the
#     "orphans" are often matched GT nodes with no split edge). Same blind spot as v2.5.
# => LB-best = pure v2 graph. Keep DIV_ENABLE=False. Gates below are retained only for documentation.
DIV_ENABLE       = False      # was True in Version 7 (LB 0.822); OFF restores the LB-best v2 graph
DIV_RADIUS_UM    = 10.0       # max mother->second-daughter distance (~p90 9.07; wider than the 8um motion gate on purpose)
MIN_ANGLE_DEG    = 75.0       # min angle between (D1-M) and (D2-M); GT p10=88deg -> 75 keeps margin, rejects same-side
MIN_DAUGHTER_LEN = 3          # second daughter must head a forward chain of >= this many nodes (main precision lever)
MIN_MOTHER_LEN   = 3          # mother must be an established track of >= this many nodes back (1 = off)
CHAIN_CAP        = 12         # cap on chain-length walk (cheap safety bound)

# model architecture -- MUST match the weights (geometry is selected by DETECTOR above)
if DETECTOR == 'v4':
    BASE = 16
    STRIDES  = ((2, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 2))   # symmetric isotropic: 64->32->16->8->4
    POOL_XY  = 4                                              # XY max-pool -> isotropic 64^3 grid (z untouched)
    POOL_MODE = 'max'                                         # preserves bright centroids (matches v4 training)
    MIN_DISTANCE = 1                                          # peak separation in POOLED voxels (1 = 1.625 um)
    REFINE_WIN   = (1, 5, 5)                                  # full-res CoM window after the x4 y,x map-back
elif DETECTOR == 'v5_base24_nores':
    BASE = 24                                                 # wider no-residual v5 detector
    STRIDES  = ((1, 2, 2), (1, 2, 2), (2, 2, 2), (2, 2, 2))   # same full-res anisotropic geometry as v1
    POOL_XY  = 1                                              # no pooling (full-res detection)
    POOL_MODE = 'max'
    # MIN_DISTANCE (3) and REFINE_WIN ((1,3,3)) keep the v2 full-res values set above
elif DETECTOR == 'v1':
    BASE = 16
    STRIDES  = ((1, 2, 2), (1, 2, 2), (2, 2, 2), (2, 2, 2))   # anisotropic: downsample xy first, z later
    POOL_XY  = 1                                              # no pooling (full-res detection)
    POOL_MODE = 'max'
    # MIN_DISTANCE (3) and REFINE_WIN ((1,3,3)) keep the v2 full-res values set above
else:
    raise ValueError(f'unknown DETECTOR {DETECTOR!r}')

# weights: None = auto-find under /kaggle/input (DETECTOR-aware exact filename order).
# For v5, prefer v5_base24_nores_best.pt; v5_UNet_res_best.pt is kept as a backwards-compatible fallback.
# To be fully explicit you can hardcode WEIGHTS_PATH to the intended .pt under /kaggle/input.
WEIGHTS_PATH = None

INPUT    = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = os.path.join(INPUT, 'test')
OUT      = '/kaggle/working/submission.csv'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__, '| DETECTOR', DETECTOR, '| BASE', BASE, '| POOL_XY', POOL_XY)

In [ ]:
# ---------------- IO + model + detection ----------------
def open_image(zarr_path):
    """Return the (T,Z,Y,X) array of an OME-Zarr sample (largest array)."""
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g)
    return best

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)   # raw logits; sigmoid at inference

def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)   # keep float32 (AMP-safe)

@torch.no_grad()
def predict_heatmap(model, vol, norm_pct=NORM_PCT):
    v = _normalize(vol, norm_pct)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def physical_nms(coords, scores, radius_um, scale=VOXEL):
    """Greedy non-max suppression in PHYSICAL (um) space: keep the strongest peak, drop others
    within radius_um. Isotropic in real units, unlike a voxel-space min_distance."""
    if len(coords) <= 1 or not radius_um:
        return coords, scores
    pts = coords * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), bool); keep = []
    for i in order:
        if killed[i]: continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True   # suppresses i itself + neighbours
    keep = np.array(keep)
    return coords[keep], scores[keep]

def pool_xy(vol, factor=POOL_XY, mode=POOL_MODE):
    """Max-pool a (Z,Y,X) volume in Y,X by `factor` (z untouched). NO-OP when factor==1 (v1/v5 full-res).
    v4 isotropic path: 256/4 = 64 -> the network sees a 1.625um-isotropic 64^3 grid. Matches v4 training."""
    if factor <= 1:
        return vol
    Z, Y, X = vol.shape
    Yc, Xc = (Y // factor) * factor, (X // factor) * factor
    v = vol[:, :Yc, :Xc].astype(np.float32).reshape(Z, Yc // factor, factor, Xc // factor, factor)
    return v.max(axis=(2, 4)) if mode == 'max' else v.mean(axis=(2, 4))

def detect_frames(model, arr, thr=HM_THR, min_distance=MIN_DISTANCE, max_peaks=MAX_PEAKS,
                  nms_um=NMS_UM, refine=REFINE, win=REFINE_WIN):
    """Per-frame detection, returns list of (N,3) coords in ORIGINAL voxel space.
    v1/v5 (POOL_XY=1): full-res heatmap -> peak_local_max -> refine -> physical NMS.
    v4 (POOL_XY>1): XY max-pool -> heatmap on 64^3 -> peaks -> map y,x back xPOOL_XY -> refine on full-res -> NMS."""
    model.eval(); frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32)          # (Z,256,256) original full-res
        hm = predict_heatmap(model, pool_xy(vol))            # pool_xy is a no-op when POOL_XY=1 (v1/v5)
        pk = peak_local_max(hm, min_distance=min_distance, threshold_abs=thr,
                            exclude_border=False, num_peaks=max_peaks)
        if len(pk) == 0:
            frames.append(np.zeros((0, 3))); continue
        scores = hm[pk[:, 0], pk[:, 1], pk[:, 2]].astype(float)
        coords = pk.astype(np.float64)
        if POOL_XY > 1:
            coords[:, 1] *= POOL_XY; coords[:, 2] *= POOL_XY  # y,x pooled voxel -> ORIGINAL resolution
        if refine:
            coords = refine_centroids(vol, coords, win=win)   # sub-pool xy precision on the full-res volume
        coords, scores = physical_nms(coords, scores, nms_um)
        frames.append(coords)
    return frames

# ---------------- Linking + post-processing (rule-based v2 stack) ----------------
@dataclass
class TrackGraph:
    node_t: np.ndarray; node_z: np.ndarray; node_y: np.ndarray; node_x: np.ndarray
    node_ids: np.ndarray; edges: np.ndarray; meta: dict
    @property
    def n_nodes(self): return len(self.node_ids)
    @property
    def n_edges(self): return len(self.edges)

def refine_centroids(vol, coords, win=(1, 3, 3)):
    """Intensity-weighted local centre-of-mass refinement on the original-resolution volume."""
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z - wz), min(Z, z + wz + 1)
        y0, y1 = max(0, y - wy), min(Y, y + wy + 1)
        x0, x1 = max(0, x - wx), min(X, x + wx + 1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    return out

def link_twopass(frames, tight_um=6.0, loose_um=8.0, vel_blend=0.5):
    node_ids = []; node_t = []; node_z = []; node_y = []; node_x = []; frame_ids = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            node_ids.append(nid); node_t.append(t); node_z.append(z); node_y.append(y); node_x.append(x)
            ids.append(nid); nid += 1
        frame_ids.append(ids)
    def _hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None] - C[ci][None]) ** 2).sum(2))
        D = np.sqrt(((pred[pi][:, None] - C[ci][None]) ** 2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = frame_ids[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz * SCALE[None, :]; C = curr * SCALE[None, :]
            pred = P + (vel_blend * prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = _hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += _hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c] - prev_xyz[p]) * SCALE
            for p, c in links:
                nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return TrackGraph(node_t=np.array(node_t, np.int64), node_z=np.array(node_z, float),
                      node_y=np.array(node_y, float), node_x=np.array(node_x, float),
                      node_ids=np.array(node_ids, np.int64),
                      edges=np.array(edges, np.int64).reshape(-1, 2), meta={})

def close_gaps(frames, g, max_gap=1, gap_dist_um=8.0):
    """Insert an interpolated node to bridge single-frame detection gaps (all GT edges are dt=1)."""
    if g.n_edges == 0:
        return g
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    ends_by_t = {}; starts_by_t = {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends_by_t.setdefault(t, []).append(nid)
        if nid not in has_in: starts_by_t.setdefault(t, []).append(nid)
    new_nodes = []; new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1)
            if not starts: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :]) ** 2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            big = thr * 1000 + 1
            cost = np.where(d <= thr, d, big)
            ri, ci = linear_sum_assignment(cost)
            used_s = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s: continue
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]; ts, zs, ys, xs = coords[s_id]
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac; yi = ye + (ys - ye) * frac; xi = xe + (xs - xe) * frac
                    nid = next_id; next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid)); new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used_s.add(c)
    if not new_nodes:
        return g
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g):
    if g.n_edges == 0: return g
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    if len(keep) == len(g.node_ids): return g
    return TrackGraph(node_t=g.node_t[keep], node_z=g.node_z[keep], node_y=g.node_y[keep],
                      node_x=g.node_x[keep], node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta)

def filter_short_tracks(g, min_len):
    if g.n_edges == 0 or min_len <= 1: return g
    parent = {int(n): int(n) for n in g.node_ids}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, t in g.edges.reshape(-1, 2):
        ra, rb = find(int(s)), find(int(t))
        if ra != rb: parent[ra] = rb
    comp = {}
    for n in g.node_ids:
        comp.setdefault(find(int(n)), []).append(int(n))
    keep = set()
    for members in comp.values():
        if len(members) >= min_len: keep.update(members)
    idx = [i for i, n in enumerate(g.node_ids) if int(n) in keep]
    keepset = set(int(g.node_ids[i]) for i in idx)
    edges = np.array([(int(s), int(t)) for s, t in g.edges.reshape(-1, 2)
                      if int(s) in keepset and int(t) in keepset], dtype=np.int64).reshape(-1, 2)
    return TrackGraph(node_t=g.node_t[idx], node_z=g.node_z[idx], node_y=g.node_y[idx],
                      node_x=g.node_x[idx], node_ids=g.node_ids[idx], edges=edges, meta=g.meta)

# ---------------- Post-hoc DIVISION detection (v3; identical to src/v3_divisions_eval.ipynb) ----------------
def _adjacency(g):
    """coords{nid:(t,z,y,x)}, out_map{nid:[children]}, in_map{nid:[parents]} for the current graph."""
    coords = {int(g.node_ids[i]): (int(g.node_t[i]), float(g.node_z[i]), float(g.node_y[i]), float(g.node_x[i]))
              for i in range(g.n_nodes)}
    out_map, in_map = defaultdict(list), defaultdict(list)
    for s, t in g.edges.reshape(-1, 2):
        out_map[int(s)].append(int(t)); in_map[int(t)].append(int(s))
    return coords, out_map, in_map

def _walk_len(start, nbr_map, cap=CHAIN_CAP):
    """Length (#nodes incl. start) of the chain following the first neighbour each step, capped."""
    n, cur, seen = 1, start, {start}
    while nbr_map.get(cur):
        nxt = nbr_map[cur][0]
        if nxt in seen: break
        seen.add(nxt); cur = nxt; n += 1
        if n >= cap: break
    return n

def detect_divisions(g, radius_um=DIV_RADIUS_UM, min_angle_deg=MIN_ANGLE_DEG,
                     min_daughter_len=MIN_DAUGHTER_LEN, min_mother_len=MIN_MOTHER_LEN, scale=SCALE):
    """Add mother->second-daughter edges (out-degree 1 -> 2) where geometry + persistence gates pass.
    Returns (new_graph, n_divisions_added). Only reads the 1:1 linked graph; adds edges, never nodes."""
    if not DIV_ENABLE or g.n_edges == 0:
        return g, 0
    coords, out_map, in_map = _adjacency(g)
    by_t = defaultdict(list)
    for nid, (t, z, y, x) in coords.items():
        by_t[t].append(nid)
    cos_thr = np.cos(np.deg2rad(min_angle_deg))   # require cos(angle) <= cos_thr  => angle >= min_angle
    new_edges = []; claimed = set()
    for m in sorted(coords):
        outs = out_map.get(m, [])
        if len(outs) != 1:                        # extend only established 1:1 continuations
            continue
        if min_mother_len > 1 and _walk_len(m, in_map) < min_mother_len:
            continue
        d1 = outs[0]
        tm, zm, ym, xm = coords[m]
        t1, z1, y1, x1 = coords[d1]
        pm = np.array([zm, ym, xm]); pd1 = np.array([z1, y1, x1])
        a = (pd1 - pm) * scale; na = np.linalg.norm(a)
        if na < 1e-6:
            continue
        best = None
        for c in by_t.get(t1, []):                # candidate second daughters live at the daughter frame
            if c == d1 or c in claimed or in_map.get(c):   # must be an ORPHAN birth (no parent yet)
                continue
            tc, zc, yc, xc = coords[c]
            pc = np.array([zc, yc, xc])
            b = (pc - pm) * scale; nb = np.linalg.norm(b)
            if nb > radius_um or nb < 1e-6:
                continue
            if float(np.dot(a, b) / (na * nb)) > cos_thr:   # angle too small (daughters not opposite)
                continue
            if _walk_len(c, out_map) < min_daughter_len:    # orphan must persist -> real daughter
                continue
            if best is None or nb < best[0]:
                best = (nb, c)
        if best is not None:
            new_edges.append((m, best[1])); claimed.add(best[1])
    if not new_edges:
        return g, 0
    edges = np.concatenate([g.edges.reshape(-1, 2), np.array(new_edges, np.int64).reshape(-1, 2)])
    g2 = TrackGraph(node_t=g.node_t, node_z=g.node_z, node_y=g.node_y, node_x=g.node_x,
                    node_ids=g.node_ids, edges=edges, meta=g.meta)
    return g2, len(new_edges)

def track_movie(model, arr):
    """Full per-movie pipeline: detect (+NMS) -> two-pass link -> close gaps -> prune -> filter short
    -> post-hoc divisions (adds mother->second-daughter edges for the 0.1 division term)."""
    frames = detect_frames(model, arr)
    g = link_twopass(frames, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND)
    if CLOSE_GAPS:     g = close_gaps(frames, g, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM)
    if PRUNE_ISOLATED: g = prune_isolated(g)
    if FILTER_MIN_LEN > 1: g = filter_short_tracks(g, FILTER_MIN_LEN)
    n_div = 0
    if DIV_ENABLE: g, n_div = detect_divisions(g)
    g.meta['n_div'] = n_div
    return g

In [ ]:
# ---------------- Load weights (auto-find; handles state_dict OR full checkpoint) ----------------
def find_weights():
    if WEIGHTS_PATH:
        return WEIGHTS_PATH
    # DETECTOR-aware order, keyed on exact filenames first. Generic '*.pt' is only the last-resort fallback.
    if DETECTOR == 'v4':
        pats = ('v4_UNet_iso_best.pt', 'unet_iso_heatmap.pt', 'unet_iso_latest.pt', 'unet_iso*.pt', '*.pt')
    elif DETECTOR == 'v5_base24_nores':
        pats = ('v5_base24_nores_best.pt', 'v5_UNet_nores_best.pt', 'v5_UNet_res_best.pt',
                'unet_res_latest.pt', 'unet_latest.pt', '*.pt')
    else:
        pats = ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt', '*.pt')
    for pat in pats:
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits:
            return hits[0]
    return None

def extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

def remap_v5_keys(sd):
    """v5_base24_nores was trained with a ConvBlock whose layers are named conv1/n1/conv2/n2 (see
    v5_resunet_train.ipynb). THIS notebook's ConvBlock is an nn.Sequential -- net.0=conv1, net.1=n1,
    net.3=conv2, net.4=n2 -- with byte-identical computation, only different parameter NAMES. Rename the
    v5 keys so they load into the Sequential block. The downs/ups/head keys contain none of these
    substrings, so the replace is safe (verified: remapped key set == this model's key set, strict load OK)."""
    ren = {'.conv1.': '.net.0.', '.n1.': '.net.1.', '.conv2.': '.net.3.', '.n2.': '.net.4.'}
    out = {}
    for k, v in sd.items():
        for a, b in ren.items():
            k = k.replace(a, b)
        out[k] = v
    return out

wpath = find_weights()
assert wpath, 'No .pt weights found under /kaggle/input -- attach the matching weights dataset for DETECTOR.'
model = UNet3D().to(device)            # BASE/STRIDES set by DETECTOR -> must match the .pt
state = extract_state(torch.load(wpath, map_location=device))
if DETECTOR == 'v5_base24_nores':
    state = remap_v5_keys(state)       # conv1/n1/conv2/n2 -> net.0/net.1/net.3/net.4 (else load_state_dict errors)
model.load_state_dict(state)
model.eval()
print('loaded weights:', wpath, '| DETECTOR', DETECTOR, '| BASE', BASE, '| STRIDES', STRIDES)

In [ ]:
# ---------------- Generate submission for ALL test datasets ----------------
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('test datasets:', len(test_zarrs))

rows = []; gid = 0; t0 = time.time(); tot_div = 0
for zp in test_zarrs:
    ds = os.path.basename(zp)[:-5]
    arr = open_image(zp)
    g = track_movie(model, arr)
    n_div = g.meta.get('n_div', 0); tot_div += n_div
    for i in range(g.n_nodes):
        rows.append((gid, ds, 'node', int(g.node_ids[i]), int(g.node_t[i]),
                     int(round(g.node_z[i])), int(round(g.node_y[i])), int(round(g.node_x[i])), -1, -1)); gid += 1
    for (s, tg) in g.edges:
        rows.append((gid, ds, 'edge', -1, -1, -1, -1, -1, int(s), int(tg))); gid += 1
    print(f'  {ds}: {g.n_nodes} nodes, {g.n_edges} edges, {n_div} divisions  ({time.time()-t0:.0f}s)')

cols = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
sub = pd.DataFrame(rows, columns=cols)
sub.to_csv(OUT, index=False)
print(f'\nwrote {OUT} {sub.shape} | {tot_div} divisions total across all test movies')
print(sub.head(6).to_string(index=False))